# Mini Project Bab 14 — Skrining Risiko Kanker Payudara

Notebook ini menjalankan alur *machine learning* end-to-end: pemahaman masalah, pemahaman dan persiapan data, komparasi tiga model, evaluasi, serialisasi, serta hubungan model dengan aplikasi Streamlit. Alur tersebut mengikuti struktur mini project dan deployment pada modul [ML-MODUL-01, hlm. 89–91].

> **Peringatan:** proyek ini hanya demonstrasi akademik, bukan diagnosis medis dan bukan pengganti tenaga kesehatan.

## 1. Persiapan Google Colab

Di Colab, sel berikut mengunduh repository dan memasang dependensi. Di komputer lokal, sel akan memakai repository yang sedang dibuka.

In [ ]:
from pathlib import Path
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPOSITORY_URL = "https://github.com/arfan-junianto/machine-learning-kuliah.git"
if IN_COLAB:
    REPOSITORY_ROOT = Path("/content/machine-learning-kuliah")
    if not REPOSITORY_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_ROOT)], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPOSITORY_ROOT = next(path for path in candidates if (path / "coursework/projects/bab-14-breast-cancer").is_dir())
PROJECT_ROOT = REPOSITORY_ROOT / "coursework/projects/bab-14-breast-cancer"
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Folder proyek:", PROJECT_ROOT)

## 2. Business Understanding

Tujuan akademik proyek adalah mengklasifikasikan tumor sebagai **jinak (0)** atau **ganas (1)** berdasarkan sepuluh karakteristik rata-rata inti sel. Karena kesalahan melewatkan kasus ganas lebih penting daripada alarm palsu, pemilihan model mengutamakan **F2-score**, yang memberi bobot lebih besar pada recall.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, confusion_matrix, roc_curve
from breast_cancer_project.data import load_project_data, split_data
from breast_cancer_project.modeling import build_model_specs, evaluate_model, select_best_model, tune_models
SEED = 42
pd.set_option("display.max_columns", 20)

## 3. Data Understanding

Dataset berasal dari Breast Cancer Wisconsin (Diagnostic) yang tersedia melalui scikit-learn. Proyek memilih sepuluh fitur `mean` agar formulir deployment tetap dapat digunakan.

In [ ]:
X, y, source_metadata = load_project_data()
print(f"Jumlah observasi: {len(X)}")
print(f"Jumlah fitur terpilih: {X.shape[1]}")
print("Missing value:", int(X.isna().sum().sum()))
display(X.head())
display(y.value_counts().rename(index={0: "jinak", 1: "ganas"}).to_frame("jumlah"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y.value_counts().sort_index().rename(index={0: "Jinak", 1: "Ganas"}).plot.bar(ax=axes[0], color=["#2a9d8f", "#e76f51"], rot=0)
axes[0].set(title="Distribusi kelas", ylabel="Jumlah observasi")
X[["mean radius", "mean texture", "mean area"]].plot.box(ax=axes[1], rot=20)
axes[1].set(title="Skala beberapa fitur", ylabel="Nilai")
plt.tight_layout(); plt.show()

## 4. Data Preparation

Data dibagi 80:20 secara terstratifikasi. Test set disimpan dan hanya dibuka setelah model dipilih melalui cross-validation agar evaluasi final tidak bocor ke proses pemilihan.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=SEED)
print("Training:", X_train.shape, y_train.value_counts(normalize=True).round(3).to_dict())
print("Test    :", X_test.shape, y_test.value_counts(normalize=True).round(3).to_dict())

## 5. Modeling dan Tuning

Logistic Regression, SVM-RBF, dan Random Forest dibandingkan dengan fold dan metrik yang sama. Grid di notebook diperkecil agar nyaman dijalankan di Colab; proyek utama tetap menyimpan eksperimen penuh. Preprocessing berada di dalam `Pipeline`, sehingga scaling dipelajari hanya dari data training pada setiap fold.

In [ ]:
model_specs = build_model_specs(random_state=SEED)
model_specs["Logistic Regression"]["param_grid"] = {"model__C": [0.1, 1.0], "model__class_weight": [None, "balanced"]}
model_specs["SVM-RBF"]["param_grid"] = {"model__estimator__C": [0.5, 1.0], "model__estimator__gamma": ["scale"], "model__estimator__class_weight": [None, "balanced"]}
model_specs["Random Forest"]["param_grid"] = {"model__n_estimators": [100, 200], "model__max_depth": [None, 10], "model__min_samples_split": [2], "model__class_weight": [None, "balanced"]}
cv_results, estimators = tune_models(X_train, y_train, cv_splits=5, random_state=SEED, model_specs=model_specs)
display(cv_results.drop(columns="best_params").sort_values("cv_f2_mean", ascending=False).round(4))

In [ ]:
selected_name = select_best_model(cv_results)
selected_model = estimators[selected_name]
print("Model terpilih berdasarkan F2 cross-validation:", selected_name)
print("Parameter terbaik:", cv_results.set_index("model").loc[selected_name, "best_params"])

## 6. Evaluation pada Test Set

Test set digunakan satu kali setelah model dipilih. Selain accuracy, evaluasi menampilkan precision, recall, F1, F2, ROC-AUC, PR-AUC, dan confusion matrix.

In [ ]:
test_metrics = evaluate_model(selected_model, X_test, y_test)
display(pd.Series({k: v for k, v in test_metrics.items() if k != "confusion_matrix"}, name="nilai").round(4).to_frame())
y_pred = selected_model.predict(X_test)
y_score = selected_model.predict_proba(X_test)[:, list(selected_model.classes_).index(1)]
matrix = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(matrix, display_labels=["Jinak", "Ganas"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
RocCurveDisplay.from_predictions(y_test, y_score, ax=axes[1], name=selected_name)
PrecisionRecallDisplay.from_predictions(y_test, y_score, ax=axes[2], name=selected_name)
axes[0].set_title("Confusion matrix"); axes[1].set_title("ROC curve"); axes[2].set_title("Precision–Recall curve")
plt.tight_layout(); plt.show()

## 7. Interpretasi Fitur

Permutation importance mengukur penurunan skor ketika satu fitur diacak. Nilai ini membantu menjelaskan perilaku model tanpa bergantung pada jenis algoritmanya.

In [ ]:
importance = permutation_importance(selected_model, X_test, y_test, scoring="f1", n_repeats=20, random_state=SEED)
importance_table = pd.DataFrame({"fitur": X_test.columns, "importance_mean": importance.importances_mean, "importance_std": importance.importances_std}).sort_values("importance_mean", ascending=False)
display(importance_table.round(4))
importance_table.sort_values("importance_mean").plot.barh(x="fitur", y="importance_mean", xerr="importance_std", figsize=(8, 5), legend=False)
plt.xlabel("Penurunan F1 saat fitur diacak"); plt.title("Permutation feature importance"); plt.tight_layout(); plt.show()

## 8. Serialisasi dan Contoh Prediksi

Model beserta pipeline dan metadata disimpan dengan `joblib`. Aplikasi Streamlit memuat bundle serupa sehingga preprocessing saat inferensi sama dengan saat pelatihan [ML-MODUL-01, hlm. 90–91].

In [ ]:
OUTPUT_DIR = Path("/content/bab14-output") if IN_COLAB else Path("/tmp/bab14-colab-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
bundle_path = OUTPUT_DIR / "model_bundle_colab.joblib"
bundle = {"model": selected_model, "model_name": selected_name, "features": list(X.columns), "target_mapping": source_metadata["target_mapping"], "test_metrics": test_metrics}
joblib.dump(bundle, bundle_path)
print("Bundle tersimpan di:", bundle_path)
sample = X_test.iloc[[0]]
prediction = int(selected_model.predict(sample)[0])
probability = float(selected_model.predict_proba(sample)[0, list(selected_model.classes_).index(1)])
display(sample)
print("Prediksi:", source_metadata["target_mapping"][prediction])
print(f"Probabilitas ganas: {probability:.2%}")

## 9. Deployment Streamlit

Notebook menunjukkan proses eksperimen dan pelatihan, sedangkan `app.py` menjadi antarmuka deployment. Secara lokal aplikasi dijalankan dengan `streamlit run app.py`. Aplikasi menerima sepuluh fitur, memvalidasi input, menjalankan pipeline yang sama, lalu menampilkan kelas dan probabilitas. Alur ini sesuai serialisasi → pemuatan model → input pengguna → `predict()` → keluaran pada Bab 14 [ML-MODUL-01, hlm. 90–91].

## 10. Kesimpulan dan Keterbatasan

Notebook membandingkan tiga algoritma secara konsisten, memilih model melalui cross-validation, dan mengevaluasinya sekali pada test set. Keterbatasannya: dataset relatif kecil, berasal dari satu domain medis, dan demonstrasi ini belum menjalani validasi klinis. Karena itu hasil hanya boleh dipakai untuk pembelajaran machine learning.